In [1]:
import pandas as pd
from datetime import date
import os
import hashlib

In [ ]:
df = pd.read_csv('./cheese_analyses.csv')
current_date = date.today().strftime("%Y-%m-%d")

## Exercise 1

### Correct 2 numeric values

- Corrected "fake" errors in `quantity` and `months` columns for two records

In [3]:
df_A = df.copy()
df_A.loc[10, 'quantity'] = 10 
df_A.loc[3, 'months'] = 3

df_A.to_csv(f"cheese_analyses_v1.0.1_{current_date}.csv")

### Add new column

- Added `maturation_stage` column that might be helpful for easier grouping

In [4]:
df_B = df_A.copy()
df_B["maturation_stage"] = df_B["months"].apply(lambda x: "fresh" if x<1 else "young" if x < 6 else "mature" if x<12 else "old")

df_B.to_csv(f"cheese_analyses_v1.1.0_{current_date}.csv")

### Remove rows with missing data

- Removed incomplete records to ensure data consistency

In [5]:
df_C = df_B.copy()
df_C.dropna(inplace=True)

df_C.to_csv(f"cheese_analyses_v2.0.0_{current_date}.csv")

## Exercise 2

In [ ]:
folders = [("raw", f"cheese_analyses.csv"),
           ("interim",""),
           ("cleaned", f"cheese_analyses_v1.0.1_{current_date}.csv"),
           ("analysis_ready", f"cheese_analyses_v1.1.0_{current_date}.csv"),
           ("analysis_ready", f"cheese_analyses_v2.0.0_{current_date}.csv")]

for entry in folders:
    # Create folder structure
    os.makedirs(f"data/{entry[0]}", exist_ok=True)
    # Create Placeholder README file
    with open(f"data/{entry[0]}/README.md", "w") as f:
        f.write(f"# {entry[0].capitalize()} Data\n\n Placeholder for {entry[0]} data for the project.\n Edit manually!")
    # Move data files to respective folders
    if entry[1] != "":
        os.replace(entry[1], f"data/{entry[0]}/{entry[1]}")
    
# Create Placeholder data_version.txt file
with open (f"data_version.txt", "w") as f:
    f.write(f"Placeholder for data versioning.\n Edit manually!")

## Exercise 3

### Create shasum files

In [9]:
files = [f"data/{entry[0]}/{entry[1]}" for entry in folders if entry[1] != ""]

def compute_sha256(file_path):
    sha256_hash = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except FileNotFoundError:
        return "File not found"
    
for file in files:
    hash_value = compute_sha256(file)
    with open(f"{file.split('.csv')[0]}.sha256", "w") as f:
        f.write(hash_value)

### Verify shasum files

In [10]:
def verify_integrity(file_path, expected_hash):
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    
    actual_hash = sha256_hash.hexdigest()
    if actual_hash == expected_hash:
        print(f"{file_path}: Integrity Verified. All is good.")
    else:
        print(f"{file_path}: Hash Mismatch! Data may be corrupted or modified.")

In [12]:
for file in files:
    file_path = file
    sha256_file = f"{file.split('.csv')[0]}.sha256"
    with open(sha256_file, "r") as f:
        expected_hash = f.read().strip()
    verify_integrity(file_path, expected_hash)

data/raw/cheese_analyses.csv: Integrity Verified. All is good.
data/cleaned/cheese_analyses_v1.0.1_2025-11-20.csv: Integrity Verified. All is good.
data/analysis_ready/cheese_analyses_v1.1.0_2025-11-20.csv: Integrity Verified. All is good.
data/analysis_ready/cheese_analyses_v2.0.0_2025-11-20.csv: Integrity Verified. All is good.


In [14]:
with open("CHANGELOG.md", "w") as f:
    f.write("Placeholder changelog file. Modify manually!\n\n")